# FUTURE_ML_01: Sales & Demand Forecasting for Businesses

This notebook demonstrates an end-to-end machine learning workflow for forecasting future sales from historical time-series data.

Dataset source: Kaggle dataset `abhishekjaiswal4896/store-sales-dataset` copied locally into `data/raw/store_sales.csv`.

Business objective: forecast sales, understand trends and seasonality, compare baseline and advanced models, and generate practical recommendations for business planning.

## 1. Imports and Project Setup

We start by importing the project utilities, standard data science libraries, and a few notebook helpers.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.append(str(PROJECT_ROOT / 'src'))

from forecasting_pipeline import (
    clean_sales_data,
    create_eda_plots,
    engineer_features,
    evaluate_models,
    generate_business_insights,
    load_sales_data,
    plot_actual_vs_predicted,
    plot_feature_importance,
    plot_future_forecast,
    recursive_forecast,
    save_outputs,
    set_plot_style,
    time_based_split,
    train_models,
)

set_plot_style()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'store_sales.csv'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data Loading

The dataset is loaded from CSV, the date column is converted into a datetime format, and the records are sorted chronologically for time-series analysis.

In [ ]:
raw_df = load_sales_data(DATA_PATH)
print(f'Raw shape: {raw_df.shape}')
display(raw_df.head())
display(raw_df.describe(include='all').transpose())

## 3. Data Cleaning

Here we remove duplicates, fill missing values, and cap extreme outliers using the IQR rule. This keeps the training signal stable without throwing away potentially useful records.

In [ ]:
cleaned_df, cleaning_summary = clean_sales_data(raw_df)
cleaning_summary_df = pd.DataFrame([cleaning_summary])
display(cleaning_summary_df)
display(cleaned_df.head())

## 4. Exploratory Data Analysis

We visualize daily sales, monthly trends, seasonality, and yearly comparisons. These charts help explain when demand rises or falls and whether the business has repeatable patterns.

In [ ]:
eda_paths = create_eda_plots(cleaned_df, FIGURE_DIR)
for title, path in eda_paths.items():
    display(Markdown(f'### {title.replace("_", " ").title()}'))
    display(Image(filename=str(path)))

## 5. Feature Engineering

To turn a time series into a supervised learning problem, we create calendar features, lag features, and rolling averages. Lag features help the model learn short-term memory, while rolling means help it understand smoother demand patterns.

In [ ]:
featured_df = engineer_features(cleaned_df)
print(f'Feature-engineered shape: {featured_df.shape}')
display(featured_df.head())
display(featured_df[['Sales', 'lag_1', 'lag_7', 'lag_30', 'rolling_mean_7', 'rolling_mean_30']].head())

## 6. Time-Based Train-Test Split

We use a chronological split instead of random shuffling so the model is always trained on the past and evaluated on the future. This is the correct way to validate forecasting systems.

In [ ]:
train_df, test_df = time_based_split(featured_df, test_days=60)
print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
print(f'Train date range: {train_df["Date"].min().date()} to {train_df["Date"].max().date()}')
print(f'Test date range: {test_df["Date"].min().date()} to {test_df["Date"].max().date()}')

## 7. Model Building

We train two models:

- `Linear Regression` as a simple baseline.
- `Random Forest Regressor` as a stronger non-linear model.

The comparison tells us whether the additional model complexity creates a meaningful forecasting improvement.

In [ ]:
trained_models = train_models(train_df)
metrics_df, predictions = evaluate_models(trained_models, test_df)
display(metrics_df)

## 8. Evaluation and Visualization

We compare the models with MAE and RMSE, then plot actual versus predicted sales for the best-performing model.

In [ ]:
best_model_name = metrics_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
actual_vs_pred_path = plot_actual_vs_predicted(
    test_data=test_df,
    prediction=predictions[best_model_name],
    output_path=FIGURE_DIR / 'actual_vs_predicted.png',
    title=f'Actual vs Predicted Sales ({best_model_name})',
)
display(Image(filename=str(actual_vs_pred_path)))

rf_importance_path = plot_feature_importance(
    trained_models['Random Forest'],
    FIGURE_DIR / 'random_forest_feature_importance.png',
)
display(Image(filename=str(rf_importance_path)))

## 9. Forecasting the Next 30 Days

The future forecast is generated recursively. For each new day, the model uses the latest available lag values, including its own previous predictions, to forecast the next step.

In [ ]:
future_forecast_df = recursive_forecast(best_model, cleaned_df, horizon=30)
future_forecast_path = plot_future_forecast(future_forecast_df, FIGURE_DIR / 'future_forecast.png')
display(future_forecast_df.head())
display(Image(filename=str(future_forecast_path)))

## 10. Business Insights and Recommendations

A forecasting project is valuable only when it leads to better business decisions. This section summarizes the key patterns we found and translates them into actions a business team can use.

In [ ]:
business_insights = generate_business_insights(cleaned_df, future_forecast_df)
for insight in business_insights:
    display(Markdown(f'- {insight}'))

## 11. Saving Models and Outputs

The final step saves the trained models, evaluation metrics, forecast CSV, and business insights so the project can be reused outside the notebook.

In [ ]:
save_outputs(
    models=trained_models,
    metrics=metrics_df,
    future_forecast=future_forecast_df,
    insights=business_insights,
    models_dir=MODELS_DIR,
    metrics_dir=METRICS_DIR,
)
display(Markdown('### Saved Files'))
display(pd.DataFrame({
    'artifact': [
        'linear_regression.joblib',
        'random_forest.joblib',
        'model_comparison.csv',
        'future_30_day_forecast.csv',
        'business_insights.md',
    ],
    'path': [
        str(MODELS_DIR / 'linear_regression.joblib'),
        str(MODELS_DIR / 'random_forest.joblib'),
        str(METRICS_DIR / 'model_comparison.csv'),
        str(METRICS_DIR / 'future_30_day_forecast.csv'),
        str(METRICS_DIR / 'business_insights.md'),
    ]
}))